# post-trade

Sanity check on a `micro-recorder` capture: every feature column is landing and rows key on
`event_ts_us`. Run any capture, Ctrl-C it (files seal on close), then run all cells.

Setup: `uv sync` in `strategies/`, pick the `.venv` kernel. An in-flight run's open file has no
footer yet — it is skipped automatically.

In [ ]:
import json
from pathlib import Path

import polars as pl

DATA = Path("../data/strat-micro-recorder/te-binance-spot-btcusdt")


def sealed(paths):
    """A live run's file has flushed row groups but no footer until close() — only a
    readable footer proves the file is done."""

    def has_footer(path):
        try:
            pl.read_parquet_metadata(path)
            return True
        except pl.exceptions.ComputeError:
            return False

    return [p for p in paths if has_footer(p)]


all_files = sorted(DATA.glob("features/date=*/*.parquet"))
feature_files = sealed(all_files)
assert feature_files, (
    f"no sealed feature files under {DATA.resolve()} — "
    f"{len(all_files)} present but unsealed, so a capture is still writing. "
    "Ctrl-C the run (the drain seals every file), then re-run this cell."
)
feature_files

In [ ]:
# The footer carries the id -> name dictionaries, so ids in the rows stay compact.
footer = pl.read_parquet_metadata(feature_files[-1])
feature_names = json.loads(footer["feature_dictionary"])
instrument_names = json.loads(footer["instrument_dictionary"])

print("strategy ", footer["strategy_id"])
print("scale    ", footer["fixed_scale"], " engine", footer["engine_version"])
print("features ", feature_names)
print("instruments", instrument_names)

In [ ]:
features = (
    pl.scan_parquet([str(p) for p in feature_files])
    .with_columns(
        pl.from_epoch("event_ts_us", time_unit="us").alias("event_ts"),
        pl.col("feature_id").replace_strict(dict(enumerate(feature_names))).alias("feature"),
        pl.col("instrument_id").replace_strict(dict(enumerate(instrument_names))).alias("instrument"),
    )
    .sort("event_ts_us")
    .collect()
)
print(features.schema)
features

In [ ]:
# Every declared feature should appear here with a sane value range. The volatilities lag the
# start of a capture — the EGARCH fit needs closed 1m candles behind it.
features.group_by("feature").agg(
    pl.len().alias("rows"),
    pl.col("value").min().alias("min"),
    pl.col("value").max().alias("max"),
).sort("feature")

In [ ]:
# One row per tick, one column per instrument+feature — the shape a model would read. Columns
# come from the combos the capture actually produced, in footer-dictionary order; hyphens become
# underscores so every name is a plain identifier. A combo the capture never produced never
# appears at all.
observed = set(features.select("instrument", "feature").unique().rows())
columns = [
    f"{instrument.replace('-', '_')}_{feature}"
    for instrument in instrument_names
    for feature in feature_names
    if (instrument, feature) in observed
]
# aggregate_function=None makes polars RAISE if a (tick, instrument, feature) duplicate exists —
# "first" would silently swallow it, and a duplicate-emission bug must fail loudly here.
wide = features.with_columns(
    (pl.col("instrument").str.replace_all("-", "_") + "_" + pl.col("feature")).alias("column")
).pivot("column", index="event_ts", values="value", aggregate_function=None)
wide = wide.select("event_ts", *columns).sort("event_ts")
wide

In [ ]:
# CSV for Excel. Space-separated datetimes to 3dp — Excel parses that; the ISO "T" form it imports
# as text. Nulls become empty cells. Lands beside the run it came from, under ../data/, which
# .gitignore keeps out of the repo.
export = wide
out = DATA / "wide.csv"
export.write_csv(out, datetime_format="%Y-%m-%d %H:%M:%S%.3f")

print(f"{out.resolve()}\n{export.height:,} rows x {export.width} cols, {out.stat().st_size / 1e6:.1f} MB")
if export.height > 1_048_576:
    print("WARNING: over Excel's row limit — it will silently truncate. Filter or downsample first.")

In [ ]:
rotation_files = sealed(sorted(DATA.glob("rotations/date=*/*.parquet")))
pl.read_parquet([str(p) for p in rotation_files]).sort("received_ts_us")